BLOCK 1: SETUP AND IMPORTS

In [1]:

import requests
import pandas as pd
import json
import time
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")


Libraries imported successfully!


BLOCK 2: CONFIGURATION

In [2]:
# European countries
COUNTRIES = {
    "Germany": "DE",
    "France": "FR",
    "Spain": "ES",
    "Italy": "IT",
    "Netherlands": "NL",
    "Poland": "PL",
    "Austria": "AT",
    "Bulgaria": "BG",
    "Belgium": "BE",
    "Greece": "EL",
    "Portugal": "PT",
    "Romania": "RO",
    "Sweden": "SE",
    "Denmark": "DK",
    "Czechia": "CZ",
}

# Technology types
TECH_TYPES = {
    "TOTAL": "Total",
    "HYD": "Hydro",
    "WND_ON": "Wind Onshore",
    "WND_OFF": "Wind Offshore",
    "SUN": "Solar",
    "NUC": "Nuclear",
    "GEO": "Geothermal",
    "BIO": "Biomass",
    "NGAS": "Natural Gas",
    "COAL": "Coal",
    "OIL": "Oil",
}

# Years to analyze
YEARS = [2021, 2022, 2023]

# Price consumer types
PRICE_DATASETS = {
    "nrg_pc_104": "Household Consumers",
    "nrg_pc_204": "Industry Consumers",
    "nrg_pc_304": "All Consumers",
}

print("Configuration set successfully!")
print(f"Countries: {len(COUNTRIES)}")
print(f"Technologies: {len(TECH_TYPES)}")
print(f"Years: {YEARS}")

Configuration set successfully!
Countries: 15
Technologies: 11
Years: [2021, 2022, 2023]


BLOCK 3: FETCH CAPACITY DATA

In [ ]:
def fetch_capacity_total(years=YEARS):
    """
    Fetch total installed electricity generation capacity per country per year.
    Returns DataFrame with total capacity in MW.
    """
    all_data = []
    base_url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_inf_epc"

    for year in years:
        print(f"\nFetching capacity data for {year}...")

        for country_name, country_code in COUNTRIES.items():
            try:
                params = {"geo": country_code, "time": year, "format": "JSON"}

                response = requests.get(base_url, params=params, timeout=30)

                if response.status_code != 200:
                    print(
                        f"  [SKIP] {country_name} ({year}): HTTP {response.status_code}"
                    )
                    continue

                data = response.json()

                if "value" in data and len(data["value"]) > 0:
                    # Sum all non-zero values for this country/year
                    # This gives us total installed capacity
                    total_capacity = 0
                    record_count = 0

                    for key, value in data["value"].items():
                        if value is not None and not pd.isna(value):
                            try:
                                val_float = float(value)
                                if val_float > 0:
                                    total_capacity += val_float
                                    record_count += 1
                            except:
                                continue

                    if total_capacity > 0:
                        all_data.append(
                            {
                                "country": country_name,
                                "country_code": country_code,
                                "total_capacity_mw": total_capacity,
                                "year": year,
                                "data_points": record_count,
                            }
                        )

                        print(
                            f"  [OK] {country_name} ({year}): {total_capacity:,.0f} MW"
                        )
                    else:
                        print(f"  [NO DATA] {country_name} ({year})")

                time.sleep(0.5)

            except Exception as e:
                print(f"  [ERROR] {country_name} ({year}): {str(e)}")
                continue

    if len(all_data) > 0:
        df_capacity = pd.DataFrame(all_data)
        print(f"\nCapacity data fetched: {len(df_capacity)} records")
        return df_capacity
    else:
        print("\n[ERROR] No capacity data retrieved!")
        return pd.DataFrame()


# Run the fetch
df_capacity = fetch_capacity_total()
print("\nCapacity Data Sample:")
print(df_capacity.head(15))



Fetching capacity data for 2021...
  [OK] Germany (2021): 992,280 MW
  [OK] France (2021): 737,014 MW
  [OK] Spain (2021): 518,909 MW
  [OK] Italy (2021): 606,289 MW
  [OK] Netherlands (2021): 235,937 MW
  [OK] Poland (2021): 242,205 MW
  [OK] Austria (2021): 165,678 MW
  [OK] Bulgaria (2021): 56,671 MW
  [OK] Belgium (2021): 143,518 MW
  [OK] Greece (2021): 111,799 MW
  [OK] Portugal (2021): 110,054 MW
  [OK] Romania (2021): 91,308 MW
  [OK] Sweden (2021): 218,434 MW
  [OK] Denmark (2021): 85,563 MW
  [OK] Czechia (2021): 97,166 MW

Fetching capacity data for 2022...
  [OK] Germany (2022): 1,427,609 MW
  [OK] France (2022): 793,250 MW
  [OK] Spain (2022): 666,350 MW
  [OK] Italy (2022): 668,234 MW
  [OK] Netherlands (2022): 308,532 MW
  [OK] Poland (2022): 298,191 MW
  [OK] Austria (2022): 164,247 MW
  [OK] Bulgaria (2022): 70,317 MW
  [OK] Belgium (2022): 157,774 MW
  [OK] Greece (2022): 137,980 MW
  [OK] Portugal (2022): 136,109 MW
  [OK] Romania (2022): 94,326 MW
  [OK] Sweden (20

BLOCK 3A: FETCH CAPACITY DATA PER TECHNOLOGY PER COUNTRY (FIXED)
Description: Download installed electricity generation capacity by technology type
Includes fetching dimension metadata to decode technology codes

In [17]:

def fetch_capacity_by_technology_with_metadata(years=YEARS):
    """
    Fetch installed electricity generation capacity by technology type per country per year.
    Also fetches dimension metadata to decode numeric technology codes to names.
    """
    all_data = []
    base_url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nrg_inf_epc"

    # First, fetch dimension metadata for technology codes
    print("Fetching dimension metadata...\n")

    tech_code_map = {}
    try:
        # Fetch metadata
        meta_response = requests.get(base_url, params={"format": "JSON"}, timeout=30)
        if meta_response.status_code == 200:
            meta_data = meta_response.json()

            # Try to extract dimension information
            if "dimension" in meta_data:
                dimensions = meta_data["dimension"]
                print(f"Available dimensions: {list(dimensions.keys())}\n")

                # Try different possible dimension names for technology
                for dim_name in ["nrg_tech", "technology", "tech", "TIME_PERIOD"]:
                    if dim_name in dimensions:
                        dim_data = dimensions[dim_name]
                        if isinstance(dim_data, dict):
                            for code, info in dim_data.items():
                                if isinstance(info, dict) and "label" in info:
                                    tech_code_map[code] = info["label"]
                                elif isinstance(info, str):
                                    tech_code_map[code] = info
                        print(f"Found {len(tech_code_map)} mappings in {dim_name}")
    except Exception as e:
        print(f"Could not fetch metadata: {str(e)}\n")

    if len(tech_code_map) == 0:
        print("Using complete technology code mapping...\n")
        tech_code_map = {
            # Totals and main groups
            "0": "Total",
            "1": "Fossil Fuels",
            "2": "Nuclear",
            "9": "Renewables Total",
            # Detailed fossil fuels
            "17": "Fossil - Solid",
            "18": "Coal",
            "19": "Lignite",
            "20": "Hard Coal",
            "27": "Fossil - Gas",
            "28": "Oil",
            "29": "Other Fossil",
            "30": "Peat",
            "31": "Gas - Natural Gas",
            "32": "Gas - Derived Gas",
            "33": "Liquefied Gas",
            "34": "Biogas",
            "35": "Other Gases",
            "38": "Waste",
            "39": "Industrial Waste",
            "40": "Municipal Waste",
            # Thermal plants
            "99": "Conventional Thermal",
            "100": "Thermal - Fossil Fuels",
            "101": "Thermal - Fossil Hard Coal",
            "108": "Thermal - Fossil Natural Gas",
            "109": "Thermal - Fossil Oil",
            "110": "Thermal - Fossil Brown Coal/Lignite",
            "111": "Thermal - Peat",
            "112": "Thermal - Gas Derived",
            "113": "Thermal - Biogas",
            "114": "Thermal - Biomass",
            "115": "Thermal - Waste",
            "116": "Thermal - Other Fossil",
            "117": "Thermal - Nuclear",
            "118": "Nuclear - Reactor Type A",
            "119": "Nuclear - Reactor Type B",
            "120": "Nuclear - Other",
            # Hydro
            "13": "Hydro Total",
            "37": "Hydro - Run-of-River",
            "121": "Hydro - Reservoir",
            "122": "Hydro - Pumped Storage",
            "123": "Hydro - Other",
            # Wind
            "10": "Wind Onshore",
            "11": "Wind Offshore",
            "135": "Wind - Onshore (Detailed)",
            "136": "Wind - Offshore (Detailed)",
            # Solar
            "12": "Solar",
            "54": "Solar PV",
            "55": "Solar Thermal",
            "82": "Solar - Photovoltaic",
            "83": "Solar - Thermal",
            "138": "Solar - PV (Detailed)",
            "139": "Solar - Thermal (Detailed)",
            "150": "Concentrated Solar Power",
            "151": "Solar - CSP",
            # Other renewables
            "15": "Geothermal",
            "141": "Geothermal - Electric",
            "142": "Geothermal - Heat",
            # Biomass
            "16": "Biomass",
            "45": "Biomass - Solid",
            "46": "Biomass - Gas",
            "56": "Biomass - Liquid",
            "57": "Biomass - Waste",
            "86": "Biomass - Solid Waste",
            "87": "Biomass - Gas Landfill",
            "88": "Biomass - Gas Sewage",
            "89": "Biomass - Gas Other",
            "90": "Biomass - Liquid Biofuel",
            "91": "Biomass - Biogas",
            "129": "Thermal - Renewable",
            "130": "Thermal - Renewable Biomass",
            "132": "Biomass - Solid",
            "133": "Biomass - Gas",
            # Other technologies
            "42": "Pure Hydrogen",
            "43": "Other Renewables",
            "51": "Fossil - Oil Products",
            "52": "Fossil - Shale Oil",
            "58": "Biofuel",
            "60": "Other Non-Renewable",
            "61": "Not Elsewhere Classified",
            "144": "Tidal Wave",
            "145": "Tidal",
            "146": "Wave",
            "147": "Fuel Cell",
            "148": "Ocean Thermal",
            "149": "Other Marine",
            "152": "Other Thermal",
        }

    for year in years:
        print(f"Fetching capacity by technology for {year}...")

        for country_name, country_code in COUNTRIES.items():
            try:
                params = {"geo": country_code, "time": year, "format": "JSON"}

                response = requests.get(base_url, params=params, timeout=30)

                if response.status_code != 200:
                    print(
                        f"  [SKIP] {country_name} ({year}): HTTP {response.status_code}"
                    )
                    continue

                data = response.json()

                if "value" in data and len(data["value"]) > 0:
                    valid_records = 0

                    for key, value in data["value"].items():
                        if value is not None and not pd.isna(value):
                            try:
                                val_float = float(value)
                                if val_float > 0:
                                    # The 'key' is the technology code
                                    tech_code = key
                                    tech_name = tech_code_map.get(
                                        tech_code, f"Tech {tech_code}"
                                    )

                                    all_data.append(
                                        {
                                            "country": country_name,
                                            "country_code": country_code,
                                            "technology": tech_name,
                                            "technology_code": tech_code,
                                            "capacity_mw": val_float,
                                            "year": year,
                                        }
                                    )
                                    valid_records += 1
                            except:
                                continue

                    if valid_records > 0:
                        print(
                            f"  [OK] {country_name} ({year}): {valid_records} technology records"
                        )
                    else:
                        print(f"  [NO DATA] {country_name} ({year})")

                time.sleep(0.5)

            except Exception as e:
                print(f"  [ERROR] {country_name} ({year}): {str(e)}")
                continue

    if len(all_data) > 0:
        df_capacity_tech = pd.DataFrame(all_data)
        print(f"\nCapacity by technology data fetched: {len(df_capacity_tech)} records")
        return df_capacity_tech
    else:
        print("\n[ERROR] No capacity by technology data retrieved!")
        return pd.DataFrame()


# Run the fetch
df_capacity_tech = fetch_capacity_by_technology_with_metadata()
print("\nCapacity by Technology Data Sample:")
print(df_capacity_tech.head(20))

# Check what technologies we got
print("\nUnique technologies (decoded):")
print(sorted(df_capacity_tech["technology"].unique()))


Fetching dimension metadata...

Available dimensions: ['freq', 'siec', 'plant_tec', 'operator', 'unit', 'geo', 'time']

Using complete technology code mapping...

Fetching capacity by technology for 2021...
  [OK] Germany (2021): 31 technology records
  [OK] France (2021): 68 technology records
  [OK] Spain (2021): 36 technology records
  [OK] Italy (2021): 45 technology records
  [OK] Netherlands (2021): 29 technology records
  [OK] Poland (2021): 26 technology records
  [OK] Austria (2021): 45 technology records
  [OK] Bulgaria (2021): 24 technology records
  [OK] Belgium (2021): 42 technology records
  [OK] Greece (2021): 22 technology records
  [OK] Portugal (2021): 42 technology records
  [OK] Romania (2021): 30 technology records
  [OK] Sweden (2021): 22 technology records
  [OK] Denmark (2021): 36 technology records
  [OK] Czechia (2021): 24 technology records
Fetching capacity by technology for 2022...
  [OK] Germany (2022): 59 technology records
  [OK] France (2022): 75 techno